In [0]:
customers_df = spark.read.table("samples.bakehouse.sales_customers")
franchises_df = spark.read.table("samples.bakehouse.sales_franchises")
suppliers_df = spark.read.table("samples.bakehouse.sales_suppliers")
transactions_df = spark.read.table("samples.bakehouse.sales_transactions")

transactions_df.printSchema()

In [0]:
customers_df.printSchema()

In [0]:
franchises_df.printSchema()

In [0]:
suppliers_df.printSchema()

In [0]:
enriched_transactions_df = franchises_df.join(transactions_df,on="franchiseID",how="inner")

enriched_transactions_df.display()

In [0]:
# on clause contain expression
enriched_transactions_df1 = franchises_df.join(\
    transactions_df,
    on = transactions_df.franchiseID == franchises_df.franchiseID,
    how="inner"
)
enriched_transactions_df1.display()

In [0]:
# select columns from each entity

enriched_entity_with_columns = franchises_df.select("city","country","franchiseID","name")\
    .join(transactions_df, on = transactions_df.franchiseID == franchises_df.franchiseID, how = "inner")

display(enriched_entity_with_columns)

In [0]:
# full outer join from franchise and supplier
from pyspark.sql.functions import col

full_join = franchises_df.join(\
    suppliers_df.select("supplierID",col("name").alias("supplierName"))
        , on = "supplierID", how = "full_outer")

display(full_join)

display(full_join.count())

# finiding out non matching records 
nonmatching_records = full_join.filter(col("supplierName").isNull() | col("franchiseID").isNull())\
    .select("franchiseID","name",col("supplierID").alias("orphan_supplier_Id"))

display(nonmatching_records)    

In [0]:
# set operations

franchise_suppliers = franchises_df.select("supplierID").distinct()

all_suppliers = suppliers_df.select("supplierID").distinct()

#find the supplier id that are in franchise df and not in supplier df
franchise_without_suppliers = franchise_suppliers.subtract(all_suppliers)

display(franchise_without_suppliers)

In [0]:
# all suppliers
display(franchise_suppliers.union(all_suppliers))

In [0]:
# common suppliers
display(franchise_suppliers.intersect(all_suppliers))